# BM Supermercados and Consum API client

Unlike the notebooks in `scrapers/` (Selenium + BeautifulSoup), **no page is
rendered here**: we query the JSON REST API that the **BM** and **Consum** websites
use themselves.

Both chains run on the same e-commerce platform (Aktios), so a single client works
for both; only the base URL changes:

| Chain | Search endpoint |
|---|---|
| BM | `https://www.online.bmsupermercados.es/api/rest/V1.0/catalog/product?q=...` |
| Consum | `https://tienda.consum.es/api/rest/V1.0/catalog/product?q=...` |

Parameters: `q` (search term), `limit` (max. 100 per page) and `offset` (pagination).
The API **caps results at 500** per search; very broad terms such as `pasta`, `pan`
or `aceite` hit that cap (they are flagged as `OK (500 cap)` in the summary).
The response contains `totalCount`, `hasMore` and the `products` list, with **ID, EAN,
brand, category, price, offer price and unit price** already structured.

### Ethical note (robots.txt)

Both sites' `robots.txt` include `Disallow: /api/`. This is a request, not a technical
block or a legal rule, but we state it openly. To minimise impact:

- Academic, non-commercial use.
- Low volume (~120 requests per full run) and a 1 s pause between requests.
- A `User-Agent` that identifies the project instead of impersonating a browser.

### Outputs

- `data/raw/bm_products_raw.csv` and `data/raw/consum_products_raw.csv`: one row per
  product appearance in each search.
- `data/processed/bm_products.csv` and `data/processed/consum_products.csv`:
  deduplicated (by `product_id`) with a normalized unit price (KILO, LITRO...).

In [ ]:
import re
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests

CHAINS = {
    "BM": "https://www.online.bmsupermercados.es",
    "Consum": "https://tienda.consum.es",
}
SEARCH_PATH = "/api/rest/V1.0/catalog/product"

SEARCH_TERMS = [
    "leche", "huevos", "manzana", "plátano", "tomate", "arroz",
    "pasta", "pan", "pollo", "carne picada", "salmón", "yogur",
    "queso", "agua", "zumo", "atún", "garbanzos", "aceite",
    "detergente", "papel higiénico"
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------
TEST_MODE = False
TEST_TERMS = ["leche"]

TERMS_TO_FETCH = TEST_TERMS if TEST_MODE else SEARCH_TERMS

PAGE_SIZE = 100          # maximum the API accepts
REQUEST_PAUSE = 1.0      # seconds between requests
MAX_RETRIES = 3
TIMEOUT = 20
API_RESULT_CAP = 500     # the API never returns more than 500 results per search

HEADERS = {
    "User-Agent": "UC3M-WebAnalytics-StudentProject/1.0 (academic, non-commercial)",
    "Accept": "application/json",
}

print("Searches to run:", TERMS_TO_FETCH)

In [ ]:
current = Path.cwd().resolve()
BASE_DIR = current.parent if current.name == "api_clients" else current

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("RAW:", RAW_DIR)
print("PROCESSED:", PROCESSED_DIR)

## API requests

`get_page` makes one request with retries (exponential backoff on network errors,
429 or 5xx). `search_all` walks through every page of a term until `hasMore` is
false.

In [ ]:
session = requests.Session()
session.headers.update(HEADERS)


def get_page(base_url, term, offset):
    url = base_url + SEARCH_PATH
    params = {"q": term, "limit": PAGE_SIZE, "offset": offset}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(url, params=params, timeout=TIMEOUT)

            if response.status_code == 429 or response.status_code >= 500:
                raise requests.HTTPError(f"HTTP {response.status_code}")

            response.raise_for_status()
            return response.json()

        except (requests.RequestException, ValueError) as exc:
            if attempt == MAX_RETRIES:
                raise
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({exc}). Retrying in {wait} s...")
            time.sleep(wait)


def search_all(base_url, term):
    products = []
    offset = 0
    total = None

    while True:
        data = get_page(base_url, term, offset)
        total = data.get("totalCount", 0)
        page = data.get("products", [])
        products.extend(page)

        time.sleep(REQUEST_PAUSE)

        if not data.get("hasMore") or not page:
            break
        offset += PAGE_SIZE

    return products, total

## Flatten the JSON

Each product is a nested JSON object. We keep the useful fields:

- `priceData.prices`: a list with `PRICE` (regular price) and, when there is a
  promotion, `OFFER_PRICE`. Despite its name, `centAmount` is already **in euros**.
- `centUnitAmount`: price per reference unit, described in `unitPriceUnitType`
  (BM uses `LITRO`/`KILO`; Consum uses `1 L`, `1 Kg`...).

In [ ]:
def get_price(prices, price_id):
    for price in prices:
        if price.get("id") == price_id:
            value = price.get("value", {})
            return value.get("centAmount"), value.get("centUnitAmount")
    return None, None


def flatten_product(product, supermarket, searched_item, scraped_at):
    data = product.get("productData", {})
    price_data = product.get("priceData", {})
    prices = price_data.get("prices", [])

    regular_price, regular_unit_price = get_price(prices, "PRICE")
    offer_price, offer_unit_price = get_price(prices, "OFFER_PRICE")

    categories = product.get("categories") or []

    return {
        "searched_item": searched_item,
        "supermarket": supermarket,
        "product_id": product.get("code"),
        "ean": product.get("ean"),
        "brand": (data.get("brand") or {}).get("name"),
        "name": data.get("name"),
        "description": data.get("description"),
        "category": categories[0].get("name") if categories else None,
        "regular_price": regular_price,
        "offer_price": offer_price,
        "regular_unit_price": regular_unit_price,
        "offer_unit_price": offer_unit_price,
        "price_unit_type": price_data.get("priceUnitType"),
        "unit_price_unit_type": price_data.get("unitPriceUnitType"),
        "available": data.get("availability") == 1
                     and not data.get("temporaryOutOfStock", False),
        "url": data.get("url"),
        "image_url": data.get("imageURL"),
        "scraped_at": scraped_at,
    }

## Single-search test

Before running every search, we check with `leche` on both chains that the number of
rows matches the `totalCount` returned by the API.

In [ ]:
for supermarket, base_url in CHAINS.items():
    products, total = search_all(base_url, "leche")
    scraped_at = datetime.now().isoformat(timespec="seconds")
    df_test = pd.DataFrame(
        [flatten_product(p, supermarket, "leche", scraped_at) for p in products]
    )
    print(f"{supermarket}: totalCount={total}, rows={len(df_test)}")
    display(df_test[["name", "brand", "regular_price", "offer_price",
                     "regular_unit_price", "unit_price_unit_type"]].head(5))

## All searches

If a search still fails after the retries, it is logged as `ERROR` in the summary
and the loop moves on to the next one.

In [ ]:
all_results = {supermarket: [] for supermarket in CHAINS}
summary = []

for supermarket, base_url in CHAINS.items():
    for i, term in enumerate(TERMS_TO_FETCH, start=1):
        print(f"[{supermarket}] [{i}/{len(TERMS_TO_FETCH)}] {term}")
        started = time.monotonic()

        try:
            products, total = search_all(base_url, term)
            scraped_at = datetime.now().isoformat(timespec="seconds")
            rows = [
                flatten_product(p, supermarket, term, scraped_at)
                for p in products
            ]
            all_results[supermarket].append(pd.DataFrame(rows))
            if len(rows) != total:
                status = "INCOMPLETE"
            elif total >= API_RESULT_CAP:
                status = "OK (500 cap)"
            else:
                status = "OK"

        except Exception as exc:
            print(f"  ERROR: {type(exc).__name__}: {exc}")
            rows, total, status = [], None, "ERROR"

        summary.append({
            "supermarket": supermarket,
            "searched_item": term,
            "total_count": total,
            "rows_extracted": len(rows),
            "status": status,
            "seconds": round(time.monotonic() - started, 2),
        })

df_summary = pd.DataFrame(summary)
display(df_summary)
print(df_summary.groupby(["supermarket", "status"]).size())

In [ ]:
# --------------------------------------------------
# SAVE RAW CSV (one per chain)
# --------------------------------------------------
raw = {}

for supermarket, frames in all_results.items():
    if not frames:
        print(f"{supermarket}: no data.")
        continue

    raw[supermarket] = pd.concat(frames, ignore_index=True)
    raw_path = RAW_DIR / f"{supermarket.lower()}_products_raw.csv"
    raw[supermarket].to_csv(raw_path, index=False, encoding="utf-8-sig")

    print(f"{supermarket}: {len(raw[supermarket])} raw rows -> {raw_path}")

## Processing

1. **Effective price**: `price` is what the customer pays (the offer price when there
   is one, otherwise the regular price), matching the visible price the scrapers capture.
2. **Normalized unit**: `unitPriceUnitType` is mapped to the project's common units
   (`KILO`, `LITRO`, `UNIDAD`...). If the reference is e.g. `100 g`, the unit price
   is multiplied by 10 to express it in €/kg.
3. **Deduplication** by `product_id`, keeping the first search term the product
   appeared in (same rule as DIA and ALDI).

**Known error in BM's data:** a few products (e.g. *Salmon ahumado 75 g*, *Azafrán
hebras caja 5 g*) declare `100 GR` but the value is already €/kg, so their unit price
comes out x10. Checked against the size in the name, 113 of 125 verifiable products
are correct and 4 have this error. It is not patched by hand: it is a source error,
and the outlier filter in `compare.ipynb` removes them.

In [ ]:
UNIT_MAP = {
    # API unit: (project unit, factor to base unit)
    "kg": ("KILO", 1), "kilo": ("KILO", 1), "kilos": ("KILO", 1),
    "g": ("KILO", 0.001), "gr": ("KILO", 0.001),
    "l": ("LITRO", 1), "litro": ("LITRO", 1), "litros": ("LITRO", 1),
    "ml": ("LITRO", 0.001), "cl": ("LITRO", 0.01),
    "u": ("UNIDAD", 1), "u.": ("UNIDAD", 1), "ud": ("UNIDAD", 1),
    "uds": ("UNIDAD", 1), "unidad": ("UNIDAD", 1), "unidades": ("UNIDAD", 1),
    "docena": ("DOCENA", 1), "dc": ("DOCENA", 1),
    "lavado": ("LAVADO", 1), "lavados": ("LAVADO", 1), "lv": ("LAVADO", 1),
    "rollo": ("ROLLO", 1), "rollos": ("ROLLO", 1),
    "m": ("METRO", 1), "metro": ("METRO", 1), "metros": ("METRO", 1),
}


def normalize_unit_price(unit_price, unit_text):
    """'1 Kg' -> (price, KILO); '100 g' -> (price * 10, KILO); 'LITRO' -> (price, LITRO)."""
    if unit_price is None or pd.isna(unit_price) or not unit_text or pd.isna(unit_text):
        return None, None

    match = re.match(
        r"^\s*(\d+(?:[.,]\d+)?)?\s*([a-záéíóú.]+)\s*$",
        str(unit_text).lower()
    )
    if not match:
        return None, None

    amount = float(match.group(1).replace(",", ".")) if match.group(1) else 1.0
    mapped = UNIT_MAP.get(match.group(2)) or UNIT_MAP.get(match.group(2).rstrip("."))
    if not mapped or amount == 0:
        return None, None

    unit, factor = mapped
    return round(float(unit_price) / (amount * factor), 4), unit


def process(df):
    df = df.copy()

    has_offer = df["offer_price"].notna()
    df["price"] = df["offer_price"].where(has_offer, df["regular_price"])
    unit_price = df["offer_unit_price"].where(has_offer, df["regular_unit_price"])

    normalized = [
        normalize_unit_price(p, u)
        for p, u in zip(unit_price, df["unit_price_unit_type"])
    ]
    df["price_per_unit"] = [n[0] for n in normalized]
    df["unit"] = [n[1] for n in normalized]
    df["on_offer"] = has_offer

    df = df.drop_duplicates(subset=["product_id"]).reset_index(drop=True)

    columns = [
        "searched_item", "supermarket", "product_id", "ean", "brand", "name",
        "category", "price", "regular_price", "offer_price", "on_offer",
        "price_per_unit", "unit", "unit_price_unit_type", "available",
        "url", "image_url", "scraped_at",
    ]
    return df[columns]

In [ ]:
processed = {}

for supermarket, df_raw in raw.items():
    df_processed = process(df_raw)
    before_drop = len(df_processed)

    unknown_units = df_processed.loc[
        df_processed["unit"].isna(), "unit_price_unit_type"
    ].value_counts(dropna=False)

    df_processed = df_processed[
        df_processed["price"].notna()
        & df_processed["price_per_unit"].notna()
        & df_processed["unit"].notna()
    ].reset_index(drop=True)

    processed[supermarket] = df_processed
    processed_path = PROCESSED_DIR / f"{supermarket.lower()}_products.csv"
    df_processed.to_csv(processed_path, index=False, encoding="utf-8-sig")

    print("=" * 70)
    print(f"{supermarket}")
    print("Raw rows:", len(df_raw))
    print("Unique before dropping incomplete:", before_drop)
    print("Final products:", len(df_processed))
    print("Dropped for missing price/unit:", before_drop - len(df_processed))
    if not unknown_units.empty:
        print("Unrecognized units:")
        print(unknown_units)
    print("\nUnits:")
    print(df_processed["unit"].value_counts(dropna=False))
    print("PROCESSED saved:", processed_path)